# LAB 04 — Delta Lake Optimization: From Fragmentation to Performance

## Scenario

An e-commerce platform stores orders in Delta Lake. The ETL pipeline appends small batches — one file per micro-batch — which has caused severe file fragmentation over time.
Your job: diagnose the problem from table metadata, compact and clean the table, benchmark three clustering strategies for product-level filtering, and finally hand the decision over to the platform with Automatic Liquid Clustering.

The preparation cell generates **~2.4 million synthetic orders across ~300 small files**. After compaction the data fits in only a few files, so the benchmark focuses on *how many files contain the filtered product* rather than on raw timings.

## Key Concepts

### DESCRIBE DETAIL
One row of storage metadata for a Delta table: `numFiles`, `sizeInBytes`, `clusteringColumns`, `partitionColumns`. Your before/after measuring stick for every optimization step.

### DESCRIBE HISTORY
One row per table version: `version`, `timestamp`, `operation`, `operationMetrics`. Every `OPTIMIZE`, `VACUUM START/END`, `WRITE`, and `CLUSTER BY` lands here — it is how you *prove* an operation ran.

### OPTIMIZE
Merges small Parquet files into ~1 GB targets. Safe (queries keep working), idempotent, transactional (a new version in history). Old files stay on storage for time travel until VACUUM removes them.

```sql
OPTIMIZE catalog.schema.table_name
OPTIMIZE catalog.schema.table_name ZORDER BY (col)
```

### VACUUM
Permanently deletes files no longer referenced within the retention window (default 168h). `DRY RUN` previews; `RETAIN 0 HOURS` is lab-only — it breaks time travel and needs the safety check disabled.

### Z-ORDER vs Liquid Clustering
Both co-locate similar values so file-level min/max stats can skip files:
- **Z-ORDER** — part of `OPTIMIZE`, rewrites everything each run, good for 1–2 stable filter columns
- **Liquid Clustering** — `ALTER TABLE ... CLUSTER BY (col)` + `OPTIMIZE`, incremental, keys can change later without a full rewrite. The modern default.

### CLUSTER BY AUTO (Automatic Liquid Clustering)
`ALTER TABLE ... CLUSTER BY AUTO` — predictive optimization observes the table's query patterns and picks (and evolves) the clustering columns itself. Recorded as table property `clusterByAuto = true`; the chosen columns appear in `clusteringColumns` asynchronously, so an empty list right after enabling is normal.

## Prerequisites

- Complete the `04 — Delta Optimization` demo first
- Run **Setup** and the **Preparation** cell — wait for the table generation to finish before Task 1
- Task 3 / Benchmark: files read vs pruned are shown in the **Query Profile** (serverless) or **Spark UI → SQL tab** (classic compute)
- Validation cells rely on variables from earlier tasks (`files_before`, `df_detail`, `df_history`) — run tasks in order

## Tasks Overview

| Task | Topic | What you will do |
|------|-------|-----------------|
| 1 | Inspect | Baseline the fragmented table with `DESCRIBE DETAIL` + `DESCRIBE HISTORY` |
| 2 | Compact | `OPTIMIZE`, preview + execute `VACUUM`, verify both in history |
| 3 | Compare | Apply Z-ORDER and Liquid Clustering to clones; benchmark data skipping |
| 4 | Automate | Enable `CLUSTER BY AUTO` and verify the `clusterByAuto` property |

## Hints — Tasks 1–2

### Task 1: Inspect table metrics
Both commands are one-liners around `spark.sql(...)`:
- `df_detail = spark.sql(f"DESCRIBE DETAIL {TABLE_BASE}")` — then `display(df_detail.select("format", "numFiles", "sizeInBytes", "clusteringColumns", "partitionColumns"))`
- `df_history = spark.sql(f"DESCRIBE HISTORY {TABLE_BASE} LIMIT 5")` — then display `version`, `timestamp`, `operation`, `operationMetrics`

The validation reads `df_detail` and `df_history`, so keep those exact variable names. Expect ~300 files — that IS the problem.

---

### Task 2: OPTIMIZE + VACUUM
Three statements, in order:
1. `OPTIMIZE {TABLE_BASE}` — plain, no ZORDER yet (that is Task 3)
2. `VACUUM {TABLE_BASE} RETAIN {VACUUM_RETAIN} DRY RUN` — `display()` it: these files *would* be deleted
3. `VACUUM {TABLE_BASE} RETAIN {VACUUM_RETAIN}` — the real thing

The provided code sets `VACUUM_RETAIN`: on classic compute it disables `spark.databricks.delta.retentionDurationCheck.enabled` and uses `0 HOURS` (lab only, never in production); on serverless that conf is locked, so it falls back to `168 HOURS` — VACUUM still runs and is logged, it just deletes nothing yet.
After step 1, run `DESCRIBE DETAIL` again into `detail_after` and print the before/after file counts. The validation also checks `DESCRIBE HISTORY` for an `OPTIMIZE` version and `VACUUM START`/`VACUUM END` entries — if they are missing, a statement did not actually run.

## Hints — Tasks 3–4

### Task 3: Z-ORDER vs Liquid Clustering
Two different mechanics for the same goal:
- **Z-ORDER table:** single command — `OPTIMIZE {TABLE_ZORDER} ZORDER BY (product_id)`
- **Liquid table:** two steps — `ALTER TABLE {TABLE_LIQUID} CLUSTER BY (product_id)` (metadata only), then `OPTIMIZE {TABLE_LIQUID}` (physical reorganization)

Sanity checks the validation performs — useful to eyeball yourself:
- `DESCRIBE DETAIL` on the liquid table → `clusteringColumns = ['product_id']`
- `DESCRIBE HISTORY` on the Z-ORDER table → the `OPTIMIZE` row's `operationParameters` mentions `product_id`

Then run the provided benchmark. `Files` is just the table's file count (compaction); data skipping shows up as **Files w/ match** being lower than `Files` for the clustered tables. Confirm files read vs pruned in the Query Profile (serverless) or Spark UI SQL tab (classic).

---

### Task 4: CLUSTER BY AUTO
One statement on the pre-cloned table:
```sql
ALTER TABLE <TABLE_AUTO> CLUSTER BY AUTO
```
Verify via `SHOW TBLPROPERTIES` — the validation expects `clusterByAuto = true`.
Do not worry if `clusteringColumns` is still empty in `DESCRIBE DETAIL`: predictive optimization chooses the columns later, from real query patterns. That asynchronous behaviour is the point of the feature.

## Summary

### The optimization decision ladder (2026)

| Situation | Answer |
|---|---|
| UC managed table, no special access pattern | Do nothing — predictive optimization handles OPTIMIZE/VACUUM |
| Known, stable filter column | Liquid Clustering `CLUSTER BY (col)` |
| Unknown / evolving patterns | `CLUSTER BY AUTO` |
| Legacy table, 1–2 filter columns, manual maintenance | `OPTIMIZE ... ZORDER BY` (recognize, don't recommend) |

### Exam Tips

- `OPTIMIZE` compacts small files (~1 GB target) and is logged as a table version; it does **not** delete old files — `VACUUM` does.
- Default VACUUM retention is **168 hours (7 days)**; shorter retention breaks time travel for the vacated versions.
- Liquid Clustering replaces **both** Hive-style partitioning and Z-ORDER; keys can be changed with `ALTER TABLE ... CLUSTER BY` without rewriting history.
- `CLUSTER BY AUTO` = Automatic Liquid Clustering driven by predictive optimization (UC managed tables).
- Data skipping works from per-file min/max statistics — clustering makes those ranges narrow, so more files can be skipped.

### If you get stuck
1. Re-read the Hint cell after the task's code cell — the exact syntax is always there.
2. The validation assert messages state precisely what is expected.
3. Full answers: `notebooks/solution/lab_04_optimization_solution.ipynb` (after the lab!).